In [1]:
import os
from langchain_core.documents import Document

In [2]:
os.makedirs("data/text_files", exist_ok = True)
os.makedirs("data/pdfs", exist_ok = True)

In [3]:
import io
import fitz  # PyMuPDF
from google.cloud import storage
# from langchain.docstore.document import Document
from langchain_core.documents import Document

def ingest_arxiv_to_memory(bucket_name, prefix, limit=50):
    client = storage.Client.create_anonymous_client()
    bucket = client.bucket(bucket_name)
    blobs = bucket.list_blobs(prefix=prefix)
    
    all_docs = []
    count = 0

    for blob in blobs:
        if blob.name.endswith('.pdf'):
            print(f"Streaming into memory: {blob.name}")
            
            # 1. Download as bytes instead of a file
            pdf_bytes = blob.download_as_bytes()
            
            # 2. Open PDF from memory buffer
            with fitz.open(stream=pdf_bytes, filetype="pdf") as doc:
                text = ""
                for page in doc:
                    text += page.get_text()
                
                # 3. Wrap in LangChain Document object
                new_doc = Document(
                    page_content=text,
                    metadata={
                        "source": f"gs://{bucket_name}/{blob.name}",
                        "file_name": blob.name
                    }
                )
                all_docs.append(new_doc)
            
            count += 1
            if count >= limit:
                break
                
    return all_docs

# Usage
memory_docs = ingest_arxiv_to_memory("arxiv-dataset", "arxiv/arxiv/pdf/2401/")

d:\DataScienceProjects\Langchain-RAG-Optimized\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Streaming into memory: arxiv/arxiv/pdf/2401/2401.00001v1.pdf
Streaming into memory: arxiv/arxiv/pdf/2401/2401.00002v1.pdf
Streaming into memory: arxiv/arxiv/pdf/2401/2401.00003v1.pdf
Streaming into memory: arxiv/arxiv/pdf/2401/2401.00003v2.pdf
Streaming into memory: arxiv/arxiv/pdf/2401/2401.00003v3.pdf
Streaming into memory: arxiv/arxiv/pdf/2401/2401.00003v4.pdf
Streaming into memory: arxiv/arxiv/pdf/2401/2401.00003v5.pdf
Streaming into memory: arxiv/arxiv/pdf/2401/2401.00003v6.pdf
Streaming into memory: arxiv/arxiv/pdf/2401/2401.00004v1.pdf
Streaming into memory: arxiv/arxiv/pdf/2401/2401.00005v1.pdf
Streaming into memory: arxiv/arxiv/pdf/2401/2401.00006v1.pdf
Streaming into memory: arxiv/arxiv/pdf/2401/2401.00006v2.pdf
Streaming into memory: arxiv/arxiv/pdf/2401/2401.00006v3.pdf
Streaming into memory: arxiv/arxiv/pdf/2401/2401.00007v1.pdf
Streaming into memory: arxiv/arxiv/pdf/2401/2401.00008v1.pdf
Streaming into memory: arxiv/arxiv/pdf/2401/2401.00009v1.pdf
Streaming into memory: a

In [4]:
memory_docs

[Document(metadata={'source': 'gs://arxiv-dataset/arxiv/arxiv/pdf/2401/2401.00001v1.pdf', 'file_name': 'arxiv/arxiv/pdf/2401/2401.00001v1.pdf'}, page_content='Sector Rotation by Factor Model and Fundamental Analysis\nRunjia Yang1 and Beining Shi2\n1University of California, Davis\n2University of California, Davis\nSept 2023\nAbstract\nThis study presents an analytical approach to sector rotation, leveraging both factor models and fundamental\nmetrics. We initiate with a systematic classification of sectors, followed by an empirical investigation into\ntheir returns. Through factor analysis, the paper underscores the significance of momentum and short-term\nreversion in dictating sectoral shifts. A subsequent in-depth fundamental analysis evaluates metrics such\nas PE, PB, EV-to-EBITDA, Dividend Yield, among others. Our primary contribution lies in developing a\npredictive framework based on these fundamental indicators. The constructed models, post rigorous training,\nexhibit noteworth

In [5]:
import pandas as pd

df = pd.DataFrame([doc.page_content for doc in memory_docs], columns=['text'])
df

,text
0,Sector Rotation by Factor Model and Fundamenta...
1,"Draft version January 2, 2024\nTypeset using L..."
2,GENERATIVE INVERSE DESIGN OF METAMATERIALS WIT...
3,GENERATIVE INVERSE DESIGN OF METAMATERIALS WIT...
4,GENERATIVE INVERSE DESIGN OF METAMATERIALS WIT...
5,GENERATIVE INVERSE DESIGN OF METAMATERIALS WIT...
6,GENERATIVE INVERSE DESIGN OF METAMATERIALS WIT...
7,GENERATIVE INVERSE DESIGN OF METAMATERIALS WIT...
8,Informational non-reductionist theory of consc...
9,\n \nConsciousness as a logically consistent ...


In [6]:
from langchain_community.document_loaders import DirectoryLoader,PyMuPDFLoader, TextLoader, CSVLoader

# pdfDoc = PyMuPDFLoader('data/pdfs/Ben Wilson - Machine Learning Engineering in Action-Manning Publications (2022)(Z-Lib.io).pdf')
# docs = pdfDoc.load()
# len(docs)

doc = DirectoryLoader(
    "data/pdfs",
    loader_cls= PyMuPDFLoader,
    glob = "**/*.pdf"
)

documents = doc.load()
documents

d:\DataScienceProjects\Langchain-RAG-Optimized\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'producer': 'Acrobat Distiller 20.0 (Windows); modified using iText® 7.1.15 ©2000-2021 iText Group NV (AGPL-version)', 'creator': 'FrameMaker 16.0.1(Foxit Advanced PDF Editor)', 'creationdate': '2022-03-06T16:40:24+00:00', 'source': 'data\\pdfs\\Ben Wilson - Machine Learning Engineering in Action-Manning Publications (2022)(Z-Lib.io).pdf', 'file_path': 'data\\pdfs\\Ben Wilson - Machine Learning Engineering in Action-Manning Publications (2022)(Z-Lib.io).pdf', 'total_pages': 578, 'format': 'PDF 1.6', 'title': 'Machine Learning Engineering in Action', 'author': 'Ben Wilson', 'subject': '', 'keywords': '', 'moddate': 'D:20220322233007', 'trapped': '', 'modDate': 'D:20220322233007', 'creationDate': 'D:20220306164024Z', 'page': 0}, page_content='M A N N I N G\nBen Wilson\nIN ACTION'),
 Document(metadata={'producer': 'Acrobat Distiller 20.0 (Windows); modified using iText® 7.1.15 ©2000-2021 iText Group NV (AGPL-version)', 'creator': 'FrameMaker 16.0.1(Foxit Advanced PDF E

In [7]:
# from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_community.document_loaders import DirectoryLoader,PyMuPDFLoader, TextLoader, CSVLoader
# from pathlib import Path

# def process_all_pdf(directory):
#     all_documents = []
#     dir = Path(directory)

#     # pdf_files = list(pdf_dir.glob('**/*.pdf')) 
#     files = list(dir.glob('**/*.*')) 

#     print(f'Found {len(files)} files in the directory.')

#     loader_map = {
#         '**/*.pdf' : PyMuPDFLoader,
#         '**/*.txt' : TextLoader,
#         '**/*.csv' : CSVLoader
#     }

#     # for pdf_file in pdf_files:
#     #     # loader = PyMuPDFLoader(pdf_file)
#     #     loader = DirectoryLoader(
#     #         pdf_files
#     #     )
#     #     documents = loader.load()

#     for pattern, loader_cls in loader_map.items():
#         loader = DirectoryLoader(
#             directory,
#             glob=pattern,
#             loader_cls = loader_cls,
#             use_multithreading=True,
#             max_concurrency=4,
#             silent_errors=True # Skips corrupted files instead of reading.
#         )

#         all_documents.extend(loader.load())

#     for doc in all_documents:
#         # path = doc.metadata.get('source', 'unknown')
#         # print(path.split('\\')[-1][-3:])
#         # doc.metadata['source'] = path.split('\\')[-1]
#         # doc.metadata["file_type"] = 'pdf'
#         p = Path(doc.metadata['source'])
#         # path_str = doc.metadata.get("source", "")
#         # path_obj = Path(path_str)
        
#         # .suffix gives you '.pdf' (includes the dot)
#         # .stem gives you 'filename' (no extension)
#         doc.metadata["filetype"] = p.suffix[1:].lower()
#         doc.metadata["filename"] = p.name
    
#     # all_documents.extend(documents)
#     print(f'Total Documents Loaded: {len(documents)}')

#     return all_documents

# all_pdfs_documents = process_all_pdf('./data')

In [8]:
# from pathlib import Path
# from langchain_community.document_loaders import (
#     DirectoryLoader, 
#     PyMuPDFLoader, 
#     TextLoader, 
#     CSVLoader
# )

# def process_all_documents(directory):
#     all_documents = []
#     dir_path = Path(directory)

#     if not dir_path.exists():
#         print(f"Error: Directory {directory} not found.")
#         return []

#     # Map patterns to loaders
#     loader_map = {
#         '**/*.pdf': PyMuPDFLoader,
#         '**/*.txt': TextLoader,
#         '**/*.csv': CSVLoader
#     }

#     for pattern, loader_cls in loader_map.items():
#         print(f"Loading files matching: {pattern}...")
#         loader = DirectoryLoader(
#             str(dir_path), # Path must be string
#             glob=pattern,
#             loader_cls=loader_cls,
#             use_multithreading=True,
#             max_concurrency=4,
#             silent_errors=True 
#         )
        
#         loaded_docs = loader.load()
#         print(f"  - Loaded {len(loaded_docs)} documents.")
#         all_documents.extend(loaded_docs)

#     # Metadata enrichment
#     for doc in all_documents:
#         source_path = Path(doc.metadata.get('source', ''))
#         # Using .suffix[1:] converts '.pdf' -> 'pdf'
#         doc.metadata["filetype"] = source_path.suffix[1:].lower() if source_path.suffix else "unknown"
#         doc.metadata["filename"] = source_path.name
    
#     print(f'--- Finished! Total Chunks Loaded: {len(all_documents)} ---')
#     return all_documents

# # Execute
# all_docs = process_all_documents('./data')

In [9]:
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Create chunks.
# 1. Initialize the tokenizer for your specific model
# 'cl100k_base' is used for GPT-3.5, GPT-4, and GPT-4o
tokenizer = tiktoken.get_encoding("cl100k_base")

# 2. Define a function that takes text and returns the token count
def tiktoken_len(text):
    tokens = tokenizer.encode(text, disallowed_special=())
    return len(tokens)

def split_documents(documents, chunk_size = 700, chunk_overlap = 100):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_doc = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_doc)} chunks.")

    #Show example of chuncks.
    if split_doc:
        print(f'\nExample chunks:')
        print(f'Content: {split_doc[0].page_content[:200]}...')
        print(f'Metadata: {split_doc[0].metadata}')

    return split_doc

In [10]:
# chunks = split_documents(all_pdfs_documents)
# chunks

In [11]:
chunks = split_documents(memory_docs)
chunks

Split 50 documents into 5202 chunks.

Example chunks:
Content: Sector Rotation by Factor Model and Fundamental Analysis
Runjia Yang1 and Beining Shi2
1University of California, Davis
2University of California, Davis
Sept 2023
Abstract
This study presents an analy...
Metadata: {'source': 'gs://arxiv-dataset/arxiv/arxiv/pdf/2401/2401.00001v1.pdf', 'file_name': 'arxiv/arxiv/pdf/2401/2401.00001v1.pdf'}


[Document(metadata={'source': 'gs://arxiv-dataset/arxiv/arxiv/pdf/2401/2401.00001v1.pdf', 'file_name': 'arxiv/arxiv/pdf/2401/2401.00001v1.pdf'}, page_content='Sector Rotation by Factor Model and Fundamental Analysis\nRunjia Yang1 and Beining Shi2\n1University of California, Davis\n2University of California, Davis\nSept 2023\nAbstract\nThis study presents an analytical approach to sector rotation, leveraging both factor models and fundamental\nmetrics. We initiate with a systematic classification of sectors, followed by an empirical investigation into\ntheir returns. Through factor analysis, the paper underscores the significance of momentum and short-term\nreversion in dictating sectoral shifts. A subsequent in-depth fundamental analysis evaluates metrics such'),
 Document(metadata={'source': 'gs://arxiv-dataset/arxiv/arxiv/pdf/2401/2401.00001v1.pdf', 'file_name': 'arxiv/arxiv/pdf/2401/2401.00001v1.pdf'}, page_content='as PE, PB, EV-to-EBITDA, Dividend Yield, among others. Our primary 

# Embedding and Vector Store DB

In [12]:
    import numpy as np
    from sentence_transformers import SentenceTransformer
    import chromadb
    from chromadb.config import Settings
    import uuid
    from typing import List, Dict, Any, Tuple
    from sklearn.metrics.pairwise import cosine_similarity

In [13]:
class EmbeddingManager:
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        # Use huggingface model name for sentence embedding
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model is loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except: 
            print(f'Error loading model {self.model_name}: {e}')
            raise
    
    def generate_embeddings(self, documents):
        if not self.model:
            raise ValueError("Model not loaded.")
        print(f"Generating embeddings for {len(documents)} docuemnts")
        embeddings = self.model.encode(documents, show_progress_bar=True)
        return embeddings

    # LangChain/Ragas specifically looks for this method name
    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return self.model.encode(texts).tolist()

    # And this one for single queries
    def embed_query(self, text: str) -> List[float]:
        return self.model.encode(text).tolist()

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 339.67it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model is loaded successfully. Embedding dimension: 384


## VectorDB

In [14]:
import shutil
import gc
import time

class VectorStore:
    def __init__(self, collection_name = "pdf_documents", persistent_directory = "./data/vector_store"):
        self.collection_name = collection_name
        self.persistent_directory = persistent_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        # 1. Clear any existing references to the client/vectorstore
        if hasattr(self, 'client'):
            # Some versions of Chroma use .close(), others rely on GC
            del self.client 
        
        # Force Python to clean up the objects and release file handles
        gc.collect()
        time.sleep(1) # Give Windows a moment to release the lock

        db_path = "./data/vector_store"
        if os.path.exists(db_path):
            try:
                shutil.rmtree(db_path)
                print("Database wiped successfully.")
            except PermissionError:
                print("File still locked. Close any other notebooks or apps using this DB.")
                # Optional: Fallback to just resetting the collection via the API
                # if the folder delete fails.

        # Create persistent ChromaDB client
        os.makedirs(self.persistent_directory, exist_ok=True)
        self.client = chromadb.PersistentClient(
            path=self.persistent_directory, 
            settings=Settings(allow_reset=True, anonymized_telemetry=False)
            )

        # Get or create collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={
                'hnsw:space': 'cosine',
                'description': 'PDF document embeddings for RAG'
                }
        )

        print(f"Vector store initialized. Collection: {self.collection_name}")
        print(f"Existing documents in collection: {self.collection.count()}")

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
    
        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        # Batching configuration (Optimal for local SSDs)
        batch_size = 100 
        total_docs = len(documents)
        
        for i in range(0, total_docs, batch_size):
            batch_docs = documents[i:i + batch_size]
            batch_embeddings = embeddings[i:i + batch_size]

            ids = []
            metadatas = []
            documents_text = []
            embeddings_list = []

            for i, (doc, embedding) in enumerate(zip(batch_docs, batch_embeddings)):
                doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
                ids.append(doc_id)

                metadata = dict(doc.metadata)
                metadata['doc_index'] = i
                metadata['content_length'] = len(doc.page_content)
                metadatas.append(metadata)

                # Document conntent
                documents_text.append(doc.page_content)

                # Embeddings
                embeddings_list.append(embedding.tolist())
            
            # Add to collection
            try:
                self.collection.add(
                    ids=ids,
                    metadatas=metadatas,
                    documents=documents_text,
                    embeddings=embeddings_list
                )
                print(f"Successfully added {len(documents)} documents to vector store")
                print(f"Total documents in collection: {self.collection.count()}")
            except Exception as e:
                print(f"Error adding docuemnts to vector store: {e}")
                raise


vector_store = VectorStore()
vector_store

Database wiped successfully.
Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [15]:
import gc

def ingest_data(vector_store, embedding_manager, chunks):
    # 1. Convert to embeddings
    texts = [doc.page_content for doc in chunks]
    embeddings = embedding_manager.generate_embeddings(texts)

    # 2. Store into the database
    vector_store.add_documents(chunks, embeddings)
    
    # 3. Explicitly dereference the heavy lists
    del texts
    del embeddings
    # 'chunks' is still available if needed outside this function
    
    # 4. Optional: Suggest to Python that now is a good time to clean up   
    gc.collect()

# Call the function
ingest_data(vector_store, embedding_manager, chunks)

Generating embeddings for 5202 docuemnts


Batches: 100%|██████████| 163/163 [02:27<00:00,  1.11it/s]


Adding 5202 documents to vector store...
Successfully added 5202 documents to vector store
Total documents in collection: 100
Successfully added 5202 documents to vector store
Total documents in collection: 200
Successfully added 5202 documents to vector store
Total documents in collection: 300
Successfully added 5202 documents to vector store
Total documents in collection: 400
Successfully added 5202 documents to vector store
Total documents in collection: 500
Successfully added 5202 documents to vector store
Total documents in collection: 600
Successfully added 5202 documents to vector store
Total documents in collection: 700
Successfully added 5202 documents to vector store
Total documents in collection: 800
Successfully added 5202 documents to vector store
Total documents in collection: 900
Successfully added 5202 documents to vector store
Total documents in collection: 1000
Successfully added 5202 documents to vector store
Total documents in collection: 1100
Successfully added 520

In [16]:
# vector_store.client.reset() # Wipes all collections and data

In [17]:
# # vector_store.client.reset() # Wipes all collections and data
# # Convert the text to embeddings
# texts = [doc.page_content for doc in chunks]

# #Generate Embedding
# embeddings = embedding_manager.generate_embeddings(texts)

# #store into the vector database
# vector_store.add_documents(chunks, embeddings)


In [18]:
class RAGRetrievar:
    """Handles query based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing the documents
            embedding manager: Manager for generating query embeddings
        
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    
    def retrieve (self, query: str, top_k: int = 5, score_threshold: float=0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimun similarity score threshold

        Returns:
            List of dictionaries contaning retrived documents and metadata
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        # print('Second query embeddding', query_embedding)

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results = top_k
            )

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate (zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distane)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            
            else:
                print("No documents found")
            
            return retrieved_docs
        
        except Exception as e:
            print(f'Error during retrieval: {e}')
            raise []
        
rag_retriever = RAGRetrievar(vector_store, embedding_manager)

In [19]:
rag_retriever.retrieve("What is Computational complexity?")

Retrieving documents for query: 'What is Computational complexity?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 docuemnts


Batches: 100%|██████████| 1/1 [00:00<00:00, 67.06it/s]

Retrieved 5 documents (after filtering)


[{'id': 'doc_bfd3a41d_80',
  'content': 'is that the complexity is simply exponential in n, thanks to the assumption that\nthe polynomials are homogeneous [25]. As any system of polynomials can be\ntransformed into this form by adding a variable and homogenizing, it means\nthat the doubly exponential complexity could be avoided.\nNote that the estimation of the complexity is carried out for the worst cases,\nmeanwhile, the actual computations often ﬁnish with much lower computational\ncosts. Moreover, the algorithmic improvements are successful in facilitating the\ncomputation. Currently, the F5 algorithm is regarded as the most eﬀective one\n[26].\nThe complexity of this algorithm was studied in [25].\nThe formula of\n19',
  'metadata': {'doc_index': 80,
   'file_name': 'arxiv/arxiv/pdf/2401/2401.00019v1.pdf',
   'content_length': 679,
   'source': 'gs://arxiv-dataset/arxiv/arxiv/pdf/2401/2401.00019v1.pdf'},
  'similarity_score': 0.5232738256454468,
  'distance': 0.4767261743545532,
 

In [20]:


rag_retriever.retrieve("What is Reed frog (Hyperolius spinigularis) tadpole mortality?")

Retrieving documents for query: 'What is Reed frog (Hyperolius spinigularis) tadpole mortality?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 docuemnts


Batches: 100%|██████████| 1/1 [00:00<00:00, 77.46it/s]

Retrieved 5 documents (after filtering)


[{'id': 'doc_e61ecd65_96',
  'content': 'of Health & Family Welfare, 2010).\n25.\nShennongAlpha.\nShennongAlpha\nKnowledge:\nNMM-0016, Curcuma wenyujin Rhizome Freshly-\nsliced Cleaned Accessed: 2024-05-01. https :\n/ / shennongalpha . westlake . edu . cn / en -\nzh/knowledge/nmm-0016.\n26.\nShennongAlpha.\nShennongAlpha\nKnowledge:\nNMM-000B, Ephedra sinica Stem-herbaceous Seg-\nmented and Aquafried-honey Accessed: 2024-05-\n01. https://shennongalpha.westlake.edu.\ncn/en-zh/knowledge/nmm-000b.\n27.\nShennongAlpha.\nShennongAlpha\nKnowledge:\nNMM-0006, Ephedra equisetina vel intermedia\nvel sinica Stem-herbaceous Accessed: 2024-05-01.\nhttps://shennongalpha.westlake.edu.cn/\nen-zh/knowledge/nmm-0006.\n28.\nCatalogue of Life. Taraxacum Accessed: 2024-05-',
  'metadata': {'content_length': 696,
   'file_name': 'arxiv/arxiv/pdf/2401/2401.00020v2.pdf',
   'source': 'gs://arxiv-dataset/arxiv/arxiv/pdf/2401/2401.00020v2.pdf',
   'doc_index': 96},
  'similarity_score': 0.31641268730163574,
  

In [21]:
# answer = rag_simple("What is varying intercepts model?", rag_retriever, llm)
# print(answer)

In [22]:
# answer = rag_optimized("What is varying intercepts model?", rag_retriever)
# print(answer)

In [23]:
from langchain_ollama import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Use OllamaLLM (lighter than ChatOllama) with tuned parameters
# llm = OllamaLLM(
#     model="llama3",
#     temperature=0,
#     num_ctx=2048,      # Small window = Fast response
#     num_thread=8       # Adjust to your CPU core count
# )

# llm = OllamaLLM(
#     model="llama3.2",
#     temperature=0,
#     # PERFORMANCE TWEAKS:
#     num_ctx=2048,           # Smaller memory footprint
#     keep_alive=-1,          # Prevent reloading every 5 mins
#     num_predict=256,        # Cap the response length to stay concise
#     num_gpu=1,              # Ensure it's using the main GPU
#     repeat_penalty=1.1      # Minor quality boost without speed cost
# )

llm = OllamaLLM(
    model="llama3.2",
    temperature=0,
    # PERFORMANCE TWEAKS:
    num_ctx=4096,           # Increased from 2048! CUDA 13 handles this easily on 8GB VRAM
    keep_alive=-1,          # Keeps the model in VRAM indefinitely
    num_predict=256,        
    # CRITICAL CHANGE: 
    # Use a high number to ensure ALL layers are offloaded to your 1070 Max-Q
    num_gpu=35,             
    repeat_penalty=1.1      
)

def rag_fast(query, retriever, top_k=3):
    # Retrieve only 3 chunks to keep the prompt small
    results = retriever.retrieve(query, top_k=top_k)
    context = '\n\n'.join([doc['content'] for doc in results])
    
    # Use LCEL for minimal Python overhead
    template = "Context: {context}\n\nQuestion: {query}\n\nAnswer concisely:"
    prompt = ChatPromptTemplate.from_template(template)
    
    chain = prompt | llm | StrOutputParser()
    
    # Use .stream() if you want to see the answer as it generates!
    return chain.invoke({"context": context, "query": query})

In [24]:
answer = rag_fast("What is continuous mixture model?", rag_retriever)
print(answer)


Retrieving documents for query: 'What is continuous mixture model?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 docuemnts


Batches: 100%|██████████| 1/1 [00:00<00:00, 43.03it/s]

Retrieved 3 documents (after filtering)


A continuous mixture model refers to a probabilistic model that combines multiple underlying distributions, each with its own parameters, to represent complex data. In this context, it seems to be referring to the Mixture of Cauchy (MoC) model mentioned in the text.


In [25]:
def rag_advance(query, retriever, llm, top_k=5, min_score=0.7, return_context=False):
    """
    RAG pipeline with extra features:
    -Return answer, sources, confidence score, and optionally full context.
    """

    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return{'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    #Prepare context and sources
    context = '\n\n'.join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        # 'preview': doc['content'] + "..."
        'preview': doc['content'][:150].rsplit(' ', 1)[0] + "..."
    } for doc in results]

    confidence = max([doc['similarity_score'] for doc in results])

    # Generate answer
    prompt = f""" 
            Use the following context to answer the question concisely.
            Context: {context}
            Question:{query}
            Answer:    
            """
    response = llm.invoke(
        [prompt.format(context=context, query=query)]
    )

    output = {
        'answer': response if isinstance(response, str) else response.content,
        'sources': sources,
        'confidence': confidence
    }

    if return_context:
        output['context'] = context
    
    return output

result = rag_advance('What is varying continuous mixture model?', rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'])

Retrieving documents for query: 'What is varying continuous mixture model?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 docuemnts


Batches: 100%|██████████| 1/1 [00:00<00:00, 102.83it/s]

Retrieved 3 documents (after filtering)


Answer: The Mixture of Cauchy (MoC) model.
Sources: [{'source': 'gs://arxiv-dataset/arxiv/arxiv/pdf/2401/2401.00011v1.pdf', 'page': 'unknown', 'score': 0.3382103443145752, 'preview': 'terministic matter.\nIt has to be accounted for in the\nmodelling process.\nIn our case of learning spreading parameters, the data\nis a set of...'}, {'source': 'gs://arxiv-dataset/arxiv/arxiv/pdf/2401/2401.00029v3.pdf', 'page': 'unknown', 'score': 0.335249662399292, 'preview': 'DK instead of random Gaussian noise. Thus, the basic for-\nward process (described in Sec. 3.1) in existing generative\ndiffusion models is not...'}, {'source': 'gs://arxiv-dataset/arxiv/arxiv/pdf/2401/2401.00029v3.pdf', 'page': 'unknown', 'score': 0.3296184539794922, 'preview': 'Sec. 3.3. We finally detail the model architecture in Sec. 3.4.\n3.1. Revisiting Diffusion Models\nThe diffusion model [18, 52], which is a kind of...'}]
Confidence: 0.3382103443145752
Context Preview: terministic matter.
It has to be accounted for in the

In [30]:
from ragas.testset import TestsetGenerator
from langchain_community.chat_models import ChatOllama
from langchain_community.embeddings import OllamaEmbeddings
from ragas.run_config import RunConfig
from ragas.testset.evolutions import simple, reasoning, multi_context

# 1. Initialize raw Ollama objects
# Llama 3.3 (8B) is a fantastic all-rounder that fits in your 8GB VRAM
# generator_llm = ChatOllama(model="llama3.2", timeout=120)

# generator_llm = ChatOllama(
#     model="llama3.2", 
#     format="json", 
#     temperature=0,  # Keep it deterministic
#     timeout=180
# )

generator_llm = ChatOllama(
    model="mistral", 
    format="json", 
    temperature=0,  # Keep it deterministic
    timeout=300,        # Increase to 5 minutes for local processing
    num_ctx=4096,       # Strictly limit the context window to prevent OOM
    num_thread=4
)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# 2. Initialize the Generator
# In v0.2+, from_langchain handles the internal wrapping for you
generator = TestsetGenerator.from_langchain(
    llm=generator_llm,
    embedding_model=embeddings
)

# 3. Generate using your ArXiv documents
# Tip: Start with a small testset_size to ensure stability
# dataset = generator.generate_with_langchain_docs(
#     documents=memory_docs, 
#     testset_size=5 
# )
# Throttle the process so your 1070 doesn't choke
run_config = RunConfig(
    max_retries=5,        # Give it more chances to fix the JSON
    timeout=240,         # ArXiv papers are long; give it time
    max_workers=1,        # CRITICAL: Do not run parallel jobs on one GPU
    log_tenacity=True
)

# testset = generator.generate_with_langchain_docs(
#     documents=memory_docs,
#     testset_size=5,
#     run_config=run_config
# )
# Distribute your 5 questions across different difficulty levels
testset = generator.generate_with_langchain_docs(
    documents=memory_docs,
    testset_size=5,
    query_distribution={simple: 0.5, reasoning: 0.25, multi_context: 0.25},
    run_config=run_config
)

# 4. Export to DataFrame
df = testset.to_pandas()
# df.to_csv("local_eval_testset.csv", index=False)
df

ModuleNotFoundError: No module named 'ragas.testset.evolutions'